In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import anndata as ad
import squidpy as sq
import cellcharter as cc
import pandas as pd
import scanpy as sc
import scvi
import numpy as np
import matplotlib.pyplot as plt
from lightning.pytorch import seed_everything
import glob
from os.path import join

seed_everything(12345)
scvi.settings.seed = 12345

In [ ]:
def run_cellcharter_protein(adata, batch_key=None, n_neighs=None, n_clusters=None):
    if not isinstance(adata.X, np.ndarray):
        adata.X = adata.X.toarray()  # 稀疏转稠密

    adata.X = adata.X.astype(np.float32)
    adata.raw = adata.copy()
    for sample in adata.obs[batch_key].unique():
        adata.X[adata.obs[batch_key] == sample, :] = sc.pp.scale(adata[adata.obs[batch_key] == sample], copy=True).X
    conditions = adata.obs[batch_key].unique().tolist()
    trvae = cc.tl.TRVAE(
        adata=adata,
        condition_key=batch_key,
        conditions=conditions,
        recon_loss='mse',
        use_mmd=False,
    )
    trvae_epochs = 500
    surgery_epochs = 500

    early_stopping_kwargs = {
        "early_stopping_metric": "val_unweighted_loss",
        "threshold": 0,
        "patience": 20,
        "reduce_lr": True,
        "lr_patience": 13,
        "lr_factor": 0.1,
    }
    trvae.train(
        n_epochs=trvae_epochs,
        alpha_epoch_anneal=200,
        early_stopping_kwargs=early_stopping_kwargs,
        enable_progress_bar=True)
    adata.obsm['X_trVAE'] = trvae.get_latent(adata.X, adata.obs[batch_key])
    if n_neighs is None:
        sq.gr.spatial_neighbors(adata, library_key='Slice', coord_type='generic', delaunay=True, spatial_key='spatial', percentile=99)
    else:
        sq.gr.spatial_neighbors(adata, library_key='Slice', coord_type='grid', n_neighs=n_neighs, spatial_key='spatial')
    cc.gr.aggregate_neighbors(adata, n_layers=3, use_rep='X_trVAE', out_key='X_cellcharter', sample_key='Slice')
    gmm = cc.tl.Cluster(
        n_clusters=n_clusters, 
        random_state=12345,
        # If running on GPU
        #trainer_params=dict(accelerator='gpu', devices=1)
    )
    # Then we fit the model to the data and predict the cluster label of all cells.
    gmm.fit(adata, use_rep='X_cellcharter')
    adata.obs['cluster_cellcharter_'+str(n_clusters)] = gmm.predict(adata, use_rep='X_cellcharter')
    return adata

In [ ]:
emb_save_path = "../../../results/embeddings/CellCharter-Protein"
clust_save_path = "../../../results/clusters/CellCharter-Protein"
base_path = "../../../data/processed/"

In [ ]:
data_names = [
    "Misar-Stereo", "DBiT-RnaAtac-Atac", "Mux-VisiumHD-HE", "Embryo",
    "Misar-E13-E15-E18", "Misar-E15-E18-E18",
    "Lymph-S1-S2-S3", "Lymph-S1-S2-S2",
    "Tonsil-S1-S2-S3", "Tonsil-S1-S2-S2",
    "MB-5M", "MB-H3K4me3"
]

neighbors_map = { 
    "Misar-E13-E15-E18": 4, 
    "Misar-E15-E18-E18": 4, 
    "Lymph-S1-S2-S3":6, 
    "Lymph-S1-S2-S2":6, 
    "Tonsil-S1-S2-S3":6, 
    "Tonsil-S1-S2-S2":6, 
    "MB-5M":4, 
    "MB-H3K4me3":4,
    "DBiT-RnaAtac-Atac":8,
    "Misar-Stereo":8,
    "Mux-VisiumHD-HE":6,
    "Embryo":8
}

clusters_map = { 
    "Misar-E13-E15-E18": 14, 
    "Misar-E15-E18-E18": 14, 
    "Lymph-S1-S2-S3":10, 
    "Lymph-S1-S2-S2":10, 
    "Tonsil-S1-S2-S3":4, 
    "Tonsil-S1-S2-S2":4, 
    "MB-5M":15, 
    "MB-H3K4me3":13,
    "DBiT-RnaAtac-Atac":15,
    "Misar-Stereo":18,
    "Mux-VisiumHD-HE":18,
    "Embryo":13
}

prefix="adt"
for data_name in data_names:
    data_path = join(base_path, data_name)
    files = glob.glob(data_path+"/*/*_adt.h5ad")
    adatas = []
    if len(files) > 0:
        for file in files:
            adata = sc.read_h5ad(file)
            if "Slice" not in adata.obs.columns:
                fallback_slice = file.split("/")[-2]
                adata.obs["Slice"] = fallback_slice

            adatas.append(adata)
        adata = sc.concat(adatas)
        adata.obs["Slice"] = adata.obs["Slice"].astype("category")
        adata = run_cellcharter_protein(adata, batch_key="Slice",n_neighs=neighbors_map[data_name], n_clusters=clusters_map[data_name])
        
        os.makedirs(join(emb_save_path, data_name), exist_ok=True)
        emb = adata.obsm["X_cellcharter"]
        emb = pd.DataFrame(emb, index=adata.obs_names, columns=[i for i in range(emb.shape[1])])
        emb.to_csv(join(emb_save_path, data_name, 'df_emb.csv'))
        
        os.makedirs(join(clust_save_path, data_name), exist_ok=True)
        df_clusters = adata.obs[['cluster_cellcharter_'+str(clusters_map[data_name])]].copy()
        df_clusters.columns = ['cluster']
        df_clusters.to_csv(join(clust_save_path, data_name, 'df_cluster.csv'))


In [ ]:
base_path = "../../../data/processed/Simulations/"
data_names = ["rna-0.1-12-adt-2.1", "rna-0.2-10-adt-2.2", "rna-0.3-7-adt-2.3", "rna-0.4-5-adt-2.4", "rna-0.5-4-adt-2.5"]

prefix="adt"
for data_name in data_names:
    for seed in ["1", "2", "3"]:
        data_path = join(base_path, data_name, seed)
        files = glob.glob(data_path+"/*_adt*.h5ad")
        adatas = []
        for file in files:
            adata = sc.read_h5ad(file)
            adata.X = adata.layers["counts"]
            if "Slice" not in adata.obs.columns:
                fallback_slice = file.split("/")[-1].split(".")[0]
                print(fallback_slice)
                adata.obs["Slice"] = fallback_slice

            adatas.append(adata)
        adata = sc.concat(adatas)
        adata.obs["Slice"] = adata.obs["Slice"].astype("category")
        adata = run_cellcharter_protein(adata, batch_key="Slice",n_neighs=4, n_clusters=5)
        
        os.makedirs(join(emb_save_path, 'Simulations', data_name, seed), exist_ok=True)
        emb = adata.obsm["X_cellcharter"]
        emb = pd.DataFrame(emb, index=adata.obs_names, columns=[i for i in range(emb.shape[1])])
        emb.to_csv(join(emb_save_path, 'Simulations', data_name, seed, 'df_emb.csv'))
        
        os.makedirs(join(clust_save_path, 'Simulations', data_name, seed), exist_ok=True)
        label = adata.obs[['cluster_cellcharter_5']].copy()
        label.columns = ["cluster"]
        label.to_csv(join(clust_save_path, 'Simulations', data_name, seed, 'df_cluster.csv'))